# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides an example for loading and exploring a dataset using the `mlcroissant` library. Follow along to inspect metadata, records, and fields, and perform basic exploratory data analysis (EDA).

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure mlcroissant is installed in your environment!
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"
dataset = mlc.Dataset(croissant_url)

# Get metadata as a dict
metadata = dataset.metadata.to_json()

# Print short metadata summary
print("{}: {}".format(metadata['name'], metadata['description']))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, record sets are objects describing tabular or collection-like data. We inspect what record sets are present and list their `@id`s, fields, and columns.

In [ ]:
# Get record sets using Croissant API
record_sets = dataset.record_sets
print("Record sets found:")
for rs in record_sets:
    print(f"- @id: {rs['@id']} (name: {rs.get('name', 'n/a')})")
    fields = rs.get('field', [])
    if fields:
        print("  Fields:")
        for f in fields:
            field_id = f['@id'] if isinstance(f, dict) else f
            print(f"    - {field_id}")
    columns = rs.get('column', [])
    if columns:
        print("  Columns:")
        for c in columns:
            col_id = c['@id'] if isinstance(c, dict) else c
            print(f"    - {col_id}")
    print("")

### Example: Print sample records for a record set
To demonstrate, let's print a few records from the first record set, referenced by `@id`.

In [ ]:
# If at least one record set exists, print first 3 records with their source
if record_sets:
    example_rs_id = record_sets[0]['@id']
    print(f"Sample records from record set '@id': {example_rs_id}")
    for ix, x in enumerate(dataset.records(record_set=example_rs_id)):
        print(json.dumps(x, indent=2))
        if ix >= 2:
            break

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. Use the record set and field `@id`s obtained above.

In [ ]:
# Prepare list of record set @id's
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    print(f"Columns for record set {rs_id}:", df.columns.tolist())
    dataframes[rs_id] = df

# Show first rows of the first record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"First 5 records from record set {first_rs_id}:")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We demonstrate filtering on a numeric field, normalization, and grouping. _Recall: all field and column references use the `@id` values._

In [ ]:
# Choose numeric and group fields from the first record set
df = dataframes[first_rs_id]

# List available columns
print(f"Columns for EDA (from {first_rs_id}):", df.columns.tolist())

# Example: Let's assume age field has @id 'age' (update with your actual @id from overview if needed)
numeric_field_id = 'age' if 'age' in df.columns else df.columns[0]  # fallback to any field
group_field_id = 'sex' if 'sex' in df.columns else df.columns[1] if len(df.columns) > 1 else None

threshold = 60  # Example threshold for age
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, col_norm]].head())

    # Grouping example
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped by {group_field_id}, mean of {numeric_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

We illustrate a histogram of the numeric field and a boxplot grouped by a categorical field, using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram: Numeric field
if numeric_field_id in df.columns:
    plt.figure(figsize=(6,4))
    df[numeric_field_id].hist(bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Boxplot by group field
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"Boxplot of {numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
Summarize your key findings. The dataset enables flexible exploration of clinical and molecular factors in a rare cohort of cancer survivors. Use the Croissant schema's `@id` references for reproducible access to fields and columns.